# Physical Realizability Test: Manual vs. ML Comparison

This notebook allows you to upload a single sample (.npy file) and run two physical realizability (PR) tests:

1. **Manual method**: Uses the original PR mask (last channel of the data).
2. **Machine Learning method**: Predicts the PR mask using a trained ML model.

Finally, it compares the two masks visually and computes basic classification metrics.

If the model does not perform well, it is required to be re-trained with your own dataset.

## 1. Imports and Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from IPython.display import display
from ipywidgets import FileUpload, Dropdown, VBox
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import xgboost as xgb
from src.utils.file_paths import file_paths, TISSUE_DIMENSIONS

## 2. Define ML Model Loader and Predictor

In [2]:
class PixelMLP(nn.Module):
    def __init__(self, in_features=12, hidden_sizes=(128,64,32)):
        super().__init__()
        layers = []
        prev = in_features
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.3)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return torch.sigmoid(self.net(x))



In [3]:

def load_ml_model(model_type='xgb'):
    if model_type == 'mlp':
        device = torch.device('cpu')
        mlp = PixelMLP().to(device)
        mlp.load_state_dict(torch.load(file_paths.model_save_path / 'best_pixel_mlp.pth', map_location=device))
        mlp.eval()
        return mlp
    elif model_type == 'xgb':
        booster = xgb.Booster()
        booster.load_model(str(file_paths.model_save_path / 'pixel_xgb.json'))
        return booster
    else:
        raise ValueError(f"Unsupported model: {model_type}")

## 3. File Upload Widget

In [4]:
uploader = FileUpload(accept='.npy', multiple=False)
display(uploader)


FileUpload(value=(), accept='.npy', description='Upload')

## 4. Run Comparison Function

In [5]:
from io import BytesIO


def compare_pr_methods(uploader, model_type='xgb', threshold=0.5):
    # Load uploaded array
    if len(uploader.value) == 0:
        print("Please upload a .npy sample file.")
        return
    key = list(uploader.value.keys())[0]
    content = uploader.value[key]['content']
    arr = np.load(BytesIO(content))  # expects shape (H,W,17)

    H, W, C = arr.shape
    print(f"Loaded sample with shape: {arr.shape}")

    # Manual mask: last channel
    manual_mask = arr[..., -1]

    # ML prediction
    X = arr[..., :12].reshape(-1, 12)
    model = load_ml_model(model_type)
    if model_type == 'mlp':
        with torch.no_grad():
            probs = model(torch.tensor(X, dtype=torch.float32)).numpy().ravel()
    else:
        dmat = xgb.DMatrix(X)
        probs = model.predict(dmat)
    ml_mask = (probs > threshold).astype(int).reshape(H, W)

    # Metrics
    y_true = manual_mask.flatten()
    y_pred = ml_mask.flatten()
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")
    print("Confusion Matrix:")
    print(cm)

    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(arr[..., 0], cmap='gray')
    axes[0].set_title('Example Channel (M11)')
    axes[1].imshow(manual_mask, cmap='gray')
    axes[1].set_title('Manual PR Mask')
    axes[2].imshow(ml_mask, cmap='gray')
    axes[2].set_title('ML Predicted PR Mask')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


## 5. Choose Model and Execute

In [6]:
model_selector = Dropdown(options=['xgb', 'mlp'], value='xgb', description='Model:')
run_button = VBox([model_selector])

def on_choice_change(change):
    uploader._counter = 0  # reset so compare_pr_methods re-runs
    compare_pr_methods(uploader, model_type=change['new'])

model_selector.observe(on_choice_change, names='value')
print("Upload your .npy file above, then select the ML model to compare.")

Upload your .npy file above, then select the ML model to compare.
